In [17]:
import os

from dotenv import load_dotenv

# override=True matters: without it, load_dotenv leaves any value already in
# os.environ untouched. Edit .env mid-session and the kernel keeps the stale one.
load_dotenv(override=True)

True

In [18]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [19]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=None, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if temperature is not None:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [20]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")
add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
text

'\n{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["running"]\n  }\n}\n'

In [21]:
import json

# Clean up and parse the JSON
clean_json = json.loads(text.strip())

clean_json

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}

In [ ]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

text = chat(messages, stop_sequences=["```"])
text.strip()

'aws s3 ls\n\naws ec2 describe-instances\n\naws lambda list-functions'